In [15]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from collections import Counter
from sklearn.metrics import accuracy_score

In [16]:
# read input files
train_data = pd.read_csv('income.train.5k.csv')
dev_data = pd.read_csv('income.dev.csv')
test_data = pd.read_csv('income.test.blind.csv')

X_train = train_data.drop(columns=['id', 'target'])

Y_train = train_data['target']

X_dev = dev_data.drop(columns=['id', 'target'])

Y_dev = dev_data['target']

X_test = test_data.drop(columns=['id'])

In [17]:
# Part 2 Data Preprocessing and Feature Extraction I: Naive Binarization

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoder.fit(X_train)

X_train_binarized = encoder.transform(X_train)

X_dev_binarized = encoder.transform(X_dev)

X_test_binarized = encoder.transform(X_test)

print("==================================== Part 2 ====================================")

i = 1
while i<100:
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_binarized, Y_train)

    Y_train_pred = knn.predict(X_train_binarized)
    train_accuracy = accuracy_score(Y_train, Y_train_pred)
    train_error_rate = 1 - train_accuracy
    positive_rate_train = np.mean(np.array(Y_train_pred) == '>50K')
    
    Y_dev_pred = knn.predict(X_dev_binarized)
    dev_accuracy = accuracy_score(Y_dev, Y_dev_pred)
    dev_error_rate = 1 - dev_accuracy
    positive_rate_dev = np.mean(np.array(Y_dev_pred) == '>50K')
    
    print("k ", i , "train_err", "{:.3f}".format(train_error_rate), " ", "{:.3f}".format(positive_rate_train), " dev_err ", "{:.3f}".format(dev_error_rate), " ", "{:.3f}".format(positive_rate_dev) )
    i += 2

# Generate test data on best k
k = 67  # best k value
knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train_binarized, Y_train)

Y_test_pred = knn.predict(X_test_binarized)

output_data = test_data.copy()  
output_data['target'] = Y_test_pred  

output_data.to_csv('income.test.predicted.p2.csv', index=False)


==================================== Part 2 ====================================
k  1 train_err 0.015   0.251  dev_err  0.232   0.248
k  3 train_err 0.118   0.230  dev_err  0.179   0.215
k  5 train_err 0.145   0.225  dev_err  0.168   0.204
k  7 train_err 0.154   0.217  dev_err  0.166   0.204
k  9 train_err 0.165   0.217  dev_err  0.168   0.204
k  11 train_err 0.169   0.206  dev_err  0.168   0.188
k  13 train_err 0.170   0.201  dev_err  0.167   0.183
k  15 train_err 0.171   0.200  dev_err  0.167   0.183
k  17 train_err 0.172   0.196  dev_err  0.161   0.183
k  19 train_err 0.174   0.193  dev_err  0.162   0.184
k  21 train_err 0.173   0.190  dev_err  0.162   0.180
k  23 train_err 0.175   0.194  dev_err  0.164   0.184
k  25 train_err 0.174   0.189  dev_err  0.162   0.172
k  27 train_err 0.174   0.188  dev_err  0.165   0.179
k  29 train_err 0.176   0.184  dev_err  0.165   0.175
k  31 train_err 0.178   0.184  dev_err  0.162   0.180
k  33 train_err 0.174   0.184  dev_err  0.162   0.176
k  35 

In [18]:
cat income.test.predicted.p2.csv | python3 validate.py

Your file has passed the formatting test! :)
Your positive rate is 17.9%.
Your positive rate seems reasonable.
This does not guarantee that your prediction accuracy will be good though.


In [19]:
#
# Part 3 Data Preprocessing and Feature Extraction II: Smart Binarization

num_processor = 'passthrough' 
cat_processor = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

preprocessor = ColumnTransformer([
('num', num_processor, ['age', 'hours']),
('cat', cat_processor, ['sector','edu','marriage','occupation','race','sex','country'])
])

preprocessor.fit(X_train)
X_train_smart = preprocessor.transform(X_train)
X_dev_smart = preprocessor.transform(X_dev)


In [20]:
print("==================================== Part 3 - Smart ====================================")

i = 1
while i<100:
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_smart, Y_train)

    Y_train_pred = knn.predict(X_train_smart)
    train_accuracy = accuracy_score(Y_train, Y_train_pred)
    train_error_rate = 1 - train_accuracy
    positive_rate_train = np.mean(np.array(Y_train_pred) == '>50K')
    
    Y_dev_pred = knn.predict(X_dev_smart)
    dev_accuracy = accuracy_score(Y_dev, Y_dev_pred)
    dev_error_rate = 1 - dev_accuracy
    positive_rate_dev = np.mean(np.array(Y_dev_pred) == '>50K')

    print("k ", i , "train_err", "{:.3f}".format(train_error_rate), " ", "{:.3f}".format(positive_rate_train), " dev_err ", "{:.3f}".format(dev_error_rate), " ", "{:.3f}".format(positive_rate_dev) )
    i += 2

# Part 3 - Data Preprocessing and Feature Extraction II: Smart Binarization -  Scaling

num_processor = MinMaxScaler(feature_range=(0, 2))
cat_processor = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

preprocessor = ColumnTransformer([
('num', num_processor, ['age', 'hours']),
('cat', cat_processor, ['sector','edu','marriage','occupation','race','sex','country'])
])

preprocessor.fit(X_train)
X_train_smart_scale = preprocessor.transform(X_train)
X_dev_smart_scale = preprocessor.transform(X_dev)

==================================== Part 3 - Smart ====================================
k  1 train_err 0.015   0.251  dev_err  0.269   0.273
k  3 train_err 0.129   0.248  dev_err  0.240   0.252
k  5 train_err 0.155   0.231  dev_err  0.237   0.255
k  7 train_err 0.165   0.229  dev_err  0.231   0.233
k  9 train_err 0.184   0.220  dev_err  0.221   0.217
k  11 train_err 0.187   0.219  dev_err  0.218   0.216
k  13 train_err 0.192   0.218  dev_err  0.222   0.216
k  15 train_err 0.187   0.213  dev_err  0.214   0.208
k  17 train_err 0.197   0.215  dev_err  0.215   0.201
k  19 train_err 0.195   0.210  dev_err  0.221   0.205
k  21 train_err 0.199   0.204  dev_err  0.223   0.201
k  23 train_err 0.202   0.196  dev_err  0.219   0.199
k  25 train_err 0.205   0.198  dev_err  0.227   0.195
k  27 train_err 0.208   0.192  dev_err  0.219   0.185
k  29 train_err 0.211   0.185  dev_err  0.220   0.182
k  31 train_err 0.210   0.182  dev_err  0.219   0.185
k  33 train_err 0.213   0.179  dev_err  0.216   0.17

In [21]:
print("==================================== Part 3 - Smart Scaling ====================================")

i = 1
while i<100:
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_smart_scale, Y_train)

    Y_train_pred = knn.predict(X_train_smart_scale)
    train_accuracy = accuracy_score(Y_train, Y_train_pred)
    train_error_rate = 1 - train_accuracy
    positive_rate_train = np.mean(np.array(Y_train_pred) == '>50K')
    
    Y_dev_pred = knn.predict(X_dev_smart_scale)
    dev_accuracy = accuracy_score(Y_dev, Y_dev_pred)
    dev_error_rate = 1 - dev_accuracy
    positive_rate_dev = np.mean(np.array(Y_dev_pred) == '>50K')
    
    print("k ", i , "train_err", "{:.3f}".format(train_error_rate), " ", "{:.3f}".format(positive_rate_train), " dev_err ", "{:.3f}".format(dev_error_rate), " ", "{:.3f}".format(positive_rate_dev) )
    i += 2

#  Generate test labels
k = 41  # Best k
knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train_smart_scale, Y_train)
X_test_smart_scale = preprocessor.transform(X_test)

Y_test_pred = knn.predict(X_test_smart_scale)

output_p3 = test_data.copy() 
output_p3['target'] = Y_test_pred 

output_p3.to_csv('income.test.predicted.p3.csv', index=False)


==================================== Part 3 - Smart Scaling ====================================
k  1 train_err 0.015   0.251  dev_err  0.239   0.271
k  3 train_err 0.115   0.240  dev_err  0.191   0.259
k  5 train_err 0.137   0.241  dev_err  0.179   0.243
k  7 train_err 0.143   0.240  dev_err  0.169   0.241
k  9 train_err 0.154   0.236  dev_err  0.160   0.226
k  11 train_err 0.162   0.236  dev_err  0.164   0.208
k  13 train_err 0.164   0.235  dev_err  0.163   0.219
k  15 train_err 0.164   0.231  dev_err  0.157   0.217
k  17 train_err 0.166   0.228  dev_err  0.156   0.218
k  19 train_err 0.168   0.225  dev_err  0.162   0.210
k  21 train_err 0.171   0.223  dev_err  0.158   0.210
k  23 train_err 0.171   0.223  dev_err  0.155   0.215
k  25 train_err 0.170   0.223  dev_err  0.153   0.213
k  27 train_err 0.171   0.221  dev_err  0.156   0.208
k  29 train_err 0.170   0.220  dev_err  0.151   0.209
k  31 train_err 0.170   0.218  dev_err  0.153   0.209
k  33 train_err 0.172   0.217  dev_err  0.15

In [22]:
cat income.test.predicted.p3.csv | python3 validate.py

Your file has passed the formatting test! :)
Your positive rate is 20.8%.
Your positive rate seems reasonable.
This does not guarantee that your prediction accuracy will be good though.


In [23]:
# Part 4 - Implement your own k-Nearest Neighbor Classifiers

def knn_predict_euclidean(train_data, train_labels, test_data, k):
    predictions = []
    for i in range(len(test_data)):
        query_person = test_data[i]
        distances = np.linalg.norm(train_data - query_person, axis=1)
        k_nearest_indices = np.argpartition(distances, k)[:k]
        k_nearest_labels = train_labels[k_nearest_indices]
        majority_vote = Counter(k_nearest_labels).most_common(1)[0][0]
        predictions.append(majority_vote)
    return predictions

def knn_predict_manhattan(train_data, train_labels, test_data, k):
    predictions = []
    for i in range(len(test_data)):
        query_person = test_data[i]
        distances = np.sum(np.abs(train_data - query_person), axis=1)
        k_nearest_indices = np.argpartition(distances, k)[:k]
        k_nearest_labels = train_labels[k_nearest_indices]
        majority_vote = Counter(k_nearest_labels).most_common(1)[0][0]
        predictions.append(majority_vote)
    return predictions


In [24]:
y = knn_predict_manhattan(X_train_smart_scale, Y_train, X_dev_smart_scale, 41)
dev_accuracy = accuracy_score(Y_dev, y)
dev_error_rate = 1 - dev_accuracy
positive_rate_dev = np.mean(np.array(Y_dev_pred) == '>50K')
print(dev_error_rate, positive_rate_dev)

0.14100000000000001 0.193


In [26]:
print("==================================== Part 4 - Manhattan ====================================")

i = 1
while i<100:
    Y_train_pred = knn_predict_manhattan(X_train_smart_scale, Y_train, X_train_smart_scale, i)
    train_accuracy = accuracy_score(Y_train, Y_train_pred)
    train_error_rate = 1 - train_accuracy
    positive_rate_train = np.mean(np.array(Y_train_pred) == '>50K')
    
    Y_dev_pred = knn_predict_manhattan(X_train_smart_scale, Y_train, X_dev_smart_scale, i)
    dev_accuracy = accuracy_score(Y_dev, Y_dev_pred)
    dev_error_rate = 1 - dev_accuracy
    positive_rate_dev = np.mean(np.array(Y_dev_pred) == '>50K')
    
    print("k ", i , "train_err", "{:.3f}".format(train_error_rate), " ", "{:.3f}".format(positive_rate_train), " dev_err ", "{:.3f}".format(dev_error_rate), " ", "{:.3f}".format(positive_rate_dev) )
    i += 2

Y_test_pred = knn_predict_manhattan(X_train_smart_scale, Y_train, X_test_smart_scale, 41)

output_p4_2 = test_data.copy()
output_p4_2['target'] = Y_test_pred
output_p4_2.to_csv('income.test.predicted.p4.csv', index=False)

==================================== Part 4 - Manhattan ====================================
k  1 train_err 0.015   0.251  dev_err  0.240   0.272
k  3 train_err 0.116   0.239  dev_err  0.197   0.261
k  5 train_err 0.140   0.239  dev_err  0.176   0.250
k  7 train_err 0.145   0.240  dev_err  0.166   0.240
k  9 train_err 0.155   0.236  dev_err  0.162   0.222
k  11 train_err 0.162   0.234  dev_err  0.163   0.219
k  13 train_err 0.165   0.236  dev_err  0.165   0.223
k  15 train_err 0.168   0.228  dev_err  0.163   0.217
k  17 train_err 0.169   0.228  dev_err  0.155   0.211
k  19 train_err 0.170   0.225  dev_err  0.160   0.210
k  21 train_err 0.170   0.222  dev_err  0.166   0.212
k  23 train_err 0.171   0.223  dev_err  0.162   0.216
k  25 train_err 0.170   0.221  dev_err  0.157   0.211
k  27 train_err 0.169   0.218  dev_err  0.158   0.206
k  29 train_err 0.168   0.213  dev_err  0.160   0.206
k  31 train_err 0.170   0.212  dev_err  0.159   0.205
k  33 train_err 0.170   0.210  dev_err  0.158   

In [161]:
cat income.test.predicted.p4.csv | python3 validate.py

Your file has passed the formatting test! :)
Your positive rate is 20.9%.
Your positive rate seems reasonable.
This does not guarantee that your prediction accuracy will be good though.
